In [2]:
print('ok')

ok


In [3]:
%pwd

'c:\\Users\\ASUS\\Desktop\\Medical Chatbot\\Medical-Chatbot-GenAI\\research'

In [4]:
import os
os.chdir("../")
%pwd

'c:\\Users\\ASUS\\Desktop\\Medical Chatbot\\Medical-Chatbot-GenAI'

In [5]:
import langchain
print(langchain.__version__)

0.3.21


In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.document_loaders import  DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [7]:
#Extracting data from the PDF files

def load_pdf_data(data):
    loader =DirectoryLoader(data, 
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [8]:
extracted_data=load_pdf_data(data='Data/')

In [9]:
#extracted_data

In [10]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500, chunk_overlap=20
    )
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [27]:
text_chunks=text_split(extracted_data)
print('length of the chunks',len(text_chunks))

length of the chunks 5860


In [12]:
#text_chunks

In [12]:
from langchain.embeddings import HuggingFaceEmbeddings

In [13]:
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [14]:
embeddings=download_hugging_face_embeddings()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20540\4043855499.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
c:\Users\ASUS\Desktop\Medical Chatbot\Medical-Chatbot-GenAI\my_project_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
query_result=embeddings.embed_query("What is a heart attack?")
print('length', len(query_result))

length 384


In [ ]:
#query_result

In [88]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY')

In [20]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medicalbot"

pc.create_index(
    name=index_name,
    dimension=384, # Replace with your model dimensions
    metric="cosine", # Replace with your model metric
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    ) 
)

{
    "name": "medicalbot",
    "metric": "cosine",
    "host": "medicalbot-tqktrbu.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [ ]:
import os 
os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY
os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY

In [ ]:
#embedded each chunk and upsert the embeddings into your pinecode index
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(documents =text_chunks,embedding= embeddings, index_name=index_name)

In [91]:
# Load existing index
from langchain_pinecone import PineconeVectorStore
#embed each chunk and upsert the embeddings into your pinecode index.
docsearch = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embeddings)

In [92]:
docsearch

In [93]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [94]:
retrieved_docs=retriever.invoke("What is acne?")

In [95]:
retrieved_docs

[Document(id='c104562b-3c14-4213-83cb-f72a9fcee610', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='45eef6fb-4cb5-4923-b092-bbb5761573e9', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 38.0, 'page_label': '39', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a womanâ€™s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed.(Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM

In [ ]:
#instead if the openAi LLM, we use the OpenRouter LLM as it is a free source, we dont need to pay for it...
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key = OPENROUTER_API_KEY,
    model_name = "Llama-3.3-70b-versatile",
)

In [97]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks."
    " Use the following pieces of context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


In [98]:
question_answer_chain = create_stuff_documents_chain(
    llm, prompt
)

rag_chain = create_retrieval_chain(
    retriever, question_answer_chain
)


In [99]:
response = rag_chain.invoke({"input": "What is acne?"})
print(response["answer"])

Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria. Acne is also known as acne vulgaris, the most common skin disease, affecting nearly 17 million people in the United States.


In [100]:
response = rag_chain.invoke({"input": "What is stats?"})
print(response["answer"])

I don't know what "stats" refers to in this context, as the provided text does not mention "stats" explicitly. The text appears to discuss medical topics, such as blood counts and heart conditions. If you have more information or context about "stats", I may be able to help further.


In [101]:
response = rag_chain.invoke({"input": "What is heart attack?"})
print(response["answer"])

A heart attack, also known as myocardial infarction, is a block in the blood supply to the heart, resulting in damage to the heart muscle. This occurs when the blood flow to the heart is interrupted, causing the heart muscle to become damaged. It is commonly referred to as a heart attack.
